In [ ]:
# Experiment 12: Select generation defaults from measured evidence; write ADR-001 -> release v0.2
!pip -q install pymongo
import time, datetime
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
from pymongo import MongoClient

MODEL = "gemini-3.1-flash-lite"
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"), http_options={"timeout": 30000})
db = MongoClient(userdata.get("MONGODB_URI"), serverSelectionTimeoutMS=8000)["genai_course"]

# 1) Evidence from Experiment 11
sweep = pd.DataFrame(list(db.sampling_sweep.find({}, {"_id": 0})))
ev = sweep.groupby("config").agg(accuracy=("label", lambda s: (s == "correct").mean()),
                                 temperature=("temperature", "first"), top_p=("top_p", "first"), top_k=("top_k", "first"),
                                 latency_s=("latency_s", "mean"), output_tokens=("output_tokens", "mean")).round(3)
print("Evidence (Experiment 11):"); print(ev.sort_values("accuracy", ascending=False).to_string())

# 2) Decision rule: best accuracy -> then temperature <= 1.0 (stable, reproducible) -> then lowest latency
cand = ev[ev.accuracy == ev.accuracy.max()]
cand = cand[cand.temperature <= 1.0] if (cand.temperature <= 1.0).any() else cand
chosen_name = cand.sort_values(["temperature", "latency_s"]).index[0]
chosen = {k: (None if pd.isna(v) else float(v)) for k, v in ev.loc[chosen_name, ["temperature", "top_p", "top_k"]].items()}
chosen = {k: v for k, v in chosen.items() if v is not None}
if "top_k" in chosen: chosen["top_k"] = int(chosen["top_k"])
print("\nChosen defaults:", chosen_name, chosen)

# 3) Re-run the full golden set with the chosen defaults (release v0.2)
def ask(q, retries=4):
    for a in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents="Answer in one or two sentences: " + q,
                       config=types.GenerateContentConfig(max_output_tokens=200, **chosen))
        except Exception as e:
            print("retry", a + 1, str(e)[:50]); time.sleep(5 * (a + 1))
golden = list(db.golden_set.find({}, {"_id": 0}))
rows = []
for g in golden:
    t = time.time(); r = ask(g["question"]); a = ((r.text if r else "") or "").lower().replace("-", " ")
    rows.append({"id": g["id"], "correct": all(k.lower() in a for k in g["expected_keywords"]), "latency_s": time.time() - t})
    time.sleep(4)
res = pd.DataFrame(rows)
v01 = db.releases.find_one({"release": "v0.1"}, sort=[("_id", -1)]) or {}
v02 = {"release": "v0.2", "model": MODEL, "defaults": chosen, "date": str(datetime.date.today()), "questions": len(res),
       "accuracy": round(float(res.correct.mean()), 3), "avg_latency_s": round(float(res.latency_s.mean()), 2)}
db.releases.insert_one(dict(v02))
print("\nv0.1 accuracy:", v01.get("accuracy"), "-> v0.2 accuracy:", v02["accuracy"])

# 4) ADR-001 (architecture decision record)
adr = f"""# ADR-001: Generation defaults for the course assistant
Date: {v02['date']}  |  Status: Accepted  |  Release: v0.2

## Context
The assistant answers factual course questions. We need sampling defaults that maximise correctness and stay reproducible.

## Options considered (evidence from Experiment 11, 8 golden questions x 5 configs)
{ev[['accuracy', 'latency_s', 'output_tokens']].to_markdown()}

## Decision
Use **{chosen_name}** -> {chosen} with max_output_tokens=200 on {MODEL}.
Rule: highest accuracy, then temperature <= 1.0 for stable answers, then lowest latency.

## Consequences
- Golden-set accuracy v0.1 = {v01.get('accuracy')} -> v0.2 = {v02['accuracy']} (32 questions).
- Remaining failures are knowledge gaps (e.g. course-specific tools), not sampling problems -> addressed by RAG in Module III.
- Re-evaluate when the model or prompt changes.
"""
open("ADR-001.md", "w").write(adr); db.adrs.replace_one({"id": "ADR-001"}, {"id": "ADR-001", "text": adr}, upsert=True)
print("\n" + adr)

Evidence (Experiment 11):
                  accuracy  temperature  top_p  top_k  latency_s  output_tokens
config                                                                         
T=0.7, top_p=0.9     1.000          0.7    0.9    NaN      1.322         46.500
T=2.0, top_k=100     1.000          2.0    NaN  100.0      1.845         39.250
T=1.0, top_k=40      0.875          1.0    NaN   40.0      1.415         35.500
T=1.5, top_p=1.0     0.875          1.5    1.0    NaN      2.601         41.000
greedy T=0           0.875          0.0    NaN    NaN      2.759         37.375

Chosen defaults: T=0.7, top_p=0.9 {'temperature': 0.7, 'top_p': 0.9}

v0.1 accuracy: 0.906 -> v0.2 accuracy: 0.875

# ADR-001: Generation defaults for the course assistant
Date: 2026-11-06  |  Status: Accepted  |  Release: v0.2

## Context
The assistant answers factual course questions. We need sampling defaults that maximise correctness and stay reproducible.

## Options considered (evidence from Experiment 1